# Building Your First ML Model - Solution

**Problem Statement:** Anova Insurance wants to classify individuals as **Healthy (0)** or **Unhealthy (1)** based on health data, to optimize insurance premium pricing.

**Dataset:** Healthcare_Data_Preprocessed_FIXED.csv (10,000 rows, 23 columns)

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('Healthcare_Data_Preprocessed_FIXED.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check target distribution
print("Target Distribution:")
print(df['Target'].value_counts())
print(f"\nPercentage Unhealthy (1): {df['Target'].mean()*100:.1f}%")
print(f"Percentage Healthy   (0): {(1 - df['Target'].mean())*100:.1f}%")

## 3. Data Quality Check

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("Missing Values:")
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# Check for negative ages (data entry errors mentioned in problem statement)
neg_age = df[df['Age'] < 0]
print(f"Rows with negative Age: {len(neg_age)}")
if len(neg_age) > 0:
    print(neg_age['Age'].describe())

In [ ]:
# Check for duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution plot
fig, ax = plt.subplots(figsize=(6, 4))
df['Target'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_xticklabels(['Healthy (0)', 'Unhealthy (1)'], rotation=0)
ax.set_title('Target Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key numerical features by target
numerical_cols = ['Age', 'BMI', 'Blood_Pressure', 'Cholesterol', 'Glucose_Level',
                  'Heart_Rate', 'Sleep_Hours', 'Exercise_Hours', 'Water_Intake', 'Stress_Level']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, col in enumerate(numerical_cols):
    ax = axes[i // 5, i % 5]
    df[df['Target'] == 0][col].hist(bins=30, alpha=0.5, label='Healthy', color='green', ax=ax)
    df[df['Target'] == 1][col].hist(bins=30, alpha=0.5, label='Unhealthy', color='red', ax=ax)
    ax.set_title(col)
    ax.legend(fontsize=7)
fig.suptitle('Numerical Feature Distributions by Health Status', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs Target
cat_cols = ['Smoking', 'Alcohol', 'Diet', 'MentalHealth', 'PhysicalActivity',
            'MedicalHistory', 'Allergies']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['Target'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'])
    axes[i].set_title(col)
    axes[i].set_ylabel('% of Total')
    axes[i].legend(['Healthy', 'Unhealthy'], fontsize=7)
    axes[i].tick_params(axis='x', rotation=0)
axes[-1].set_visible(False)
fig.suptitle('Categorical Features vs Health Status', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with Target
target_corr = corr['Target'].drop('Target').abs().sort_values(ascending=False)
print("Feature Correlation with Target (absolute values):")
print(target_corr.to_string())

## 5. Data Preprocessing

In [ ]:
df_clean = df.copy()

# Handle negative ages - take absolute value
df_clean['Age'] = df_clean['Age'].abs()
print(f"Age range after fix: {df_clean['Age'].min():.0f} - {df_clean['Age'].max():.0f}")

In [ ]:
# Fill missing values with median (robust to outliers)
missing_cols = df_clean.columns[df_clean.isnull().any()].tolist()
print(f"Columns with missing values: {missing_cols}")

for col in missing_cols:
    median_val = df_clean[col].median()
    df_clean[col].fillna(median_val, inplace=True)
    print(f"  {col}: filled with median = {median_val}")

print(f"\nRemaining missing values: {df_clean.isnull().sum().sum()}")

In [ ]:
# Convert boolean columns to integers
bool_cols = df_clean.select_dtypes(include='bool').columns.tolist()
print(f"Boolean columns to convert: {bool_cols}")
for col in bool_cols:
    df_clean[col] = df_clean[col].astype(int)

print("\nData types after preprocessing:")
print(df_clean.dtypes)

## 6. Feature Selection and Train-Test Split

In [ ]:
# Separate features and target
X = df_clean.drop('Target', axis=1)
y = df_clean['Target']

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"\nFeature columns ({len(X.columns)}):")
print(list(X.columns))

In [ ]:
# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTarget distribution in train: {y_train.value_counts().to_dict()}")
print(f"Target distribution in test:  {y_test.value_counts().to_dict()}")

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling complete (StandardScaler).")

## 7. Model Building

We will train and compare three models:
1. **Logistic Regression** - Simple, interpretable baseline
2. **Decision Tree** - Non-linear, easy to interpret
3. **Random Forest** - Ensemble method, generally strong performance

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
}

results = {}

for name, model in models.items():
    # Use scaled data for Logistic Regression, unscaled for tree-based models
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob),
    }
    print(f"{name} trained successfully.")

print("\nAll models trained!")

## 8. Model Evaluation and Comparison

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    name: {
        'Accuracy': r['accuracy'],
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1 Score': r['f1'],
        'ROC AUC': r['roc_auc'],
    }
    for name, r in results.items()
}).T

comparison = comparison.round(4)
print("Model Performance Comparison:")
comparison

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(10, 5))
comparison.plot(kind='bar', ax=ax)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_xticklabels(comparison.index, rotation=15)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, (name, r) in enumerate(results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Healthy', 'Unhealthy'],
                yticklabels=['Healthy', 'Unhealthy'])
    axes[i].set_title(f'{name}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))

for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {r['roc_auc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification report for each model
for name, r in results.items():
    print(f"\n{'='*50}")
    print(f"Classification Report: {name}")
    print('='*50)
    print(classification_report(y_test, r['y_pred'], target_names=['Healthy', 'Unhealthy']))

## 9. Feature Importance (Random Forest)

In [ ]:
# Feature importance from Random Forest
rf_model = results['Random Forest']['model']
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
feat_imp.plot(kind='barh', color='steelblue')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
for feat, imp in feat_imp.sort_values(ascending=False).head(5).items():
    print(f"  {feat}: {imp:.4f}")

## 10. Best Model Selection and Final Prediction

In [ ]:
# Select the best model based on F1 Score
best_model_name = comparison['F1 Score'].idxmax()
best_score = comparison.loc[best_model_name, 'F1 Score']

print(f"Best Model: {best_model_name}")
print(f"F1 Score:   {best_score:.4f}")
print(f"Accuracy:   {comparison.loc[best_model_name, 'Accuracy']:.4f}")
print(f"ROC AUC:    {comparison.loc[best_model_name, 'ROC AUC']:.4f}")

In [ ]:
# Demonstrate prediction on a sample patient
sample = X_test.iloc[[0]]
print("Sample Patient Data:")
print(sample.to_string())

best_model = results[best_model_name]['model']
if best_model_name == 'Logistic Regression':
    sample_input = scaler.transform(sample)
else:
    sample_input = sample

prediction = best_model.predict(sample_input)[0]
probability = best_model.predict_proba(sample_input)[0]

print(f"\nPrediction: {'Unhealthy' if prediction == 1 else 'Healthy'}")
print(f"Confidence: Healthy={probability[0]:.2%}, Unhealthy={probability[1]:.2%}")
print(f"Actual:     {'Unhealthy' if y_test.iloc[0] == 1 else 'Healthy'}")

## 11. Conclusion

### Summary

| Step | What We Did |
|------|-------------|
| **EDA** | Explored distributions, checked class balance, visualized feature relationships |
| **Data Cleaning** | Fixed negative ages, imputed missing values with median, converted boolean columns |
| **Feature Engineering** | Used all 22 features (dataset was already one-hot encoded for Diet_Type & Blood_Group) |
| **Modeling** | Trained Logistic Regression, Decision Tree, and Random Forest classifiers |
| **Evaluation** | Compared models using Accuracy, Precision, Recall, F1, ROC AUC, and Confusion Matrices |

### Key Findings
- The dataset had missing values primarily in Blood_Pressure, Cholesterol, MedicalHistory, and Allergies
- Some age values were negative (data entry errors) -- handled by taking absolute values
- Feature importance analysis reveals which health factors most strongly predict health status
- The best model can be used by Anova Insurance to classify applicants and set appropriate premium rates